# End-to-End Piano Roll Generation

Full pipeline: **Flow model** (L0-L3) → **Inverse PCA** → **HMEP** (L4-L5 prediction) → **Decoder** → Piano roll

In [ ]:
#| default_exp generate

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, pickle, glob, math
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from omegaconf import DictConfig

from midi_rae.core import PatchState, HierarchicalPatchState, EncoderOutput
from midi_rae.swin import SwinDecoder, SwinMaskedEmbeddingPredictor
from midi_rae.train_flow import CrossLevelFlowModel, PerLevelFlowModel, sample_source
from midi_rae.data import EmbeddingDataset
from midi_rae.utils import load_checkpoint, binarize, cjprint

In [ ]:
#| export
def make_grid_pos(n_patches, device='cpu'):
    """Construct (N, 2) grid position tensor for a square patch grid."""
    g = int(n_patches ** 0.5)
    rows, cols = torch.meshgrid(torch.arange(g), torch.arange(g), indexing='ij')
    return torch.stack([rows.flatten(), cols.flatten()], dim=1).float().to(device)

def inverse_pca_level(pca, flat_vec, n_patches, device):
    """flat (n_patches*n_components,) → (1, n_patches, orig_dim) tensor"""
    n_comp = pca.n_components_
    pca_codes = flat_vec.reshape(n_patches, n_comp).cpu().numpy()
    emb_np = pca.inverse_transform(pca_codes)
    return torch.tensor(emb_np, dtype=torch.float32, device=device).unsqueeze(0)

def build_patch_states(flow_vec, level_dims, device, pca_models=None, n_patches_list=None):
    """Convert flat flow output into a list of PatchState, one per level.
    pca_models: optional dict {i: sklearn PCA}. If None or level missing, embeddings
                are treated as already in full space (raw mode). n_patches_list must
                be provided in raw mode so we know how to reshape each level."""
    states, offset = [], 0
    for i, level_dim in enumerate(level_dims):
        if pca_models is not None and i in pca_models:
            n_patches = level_dim // pca_models[i].n_components_
            emb = inverse_pca_level(pca_models[i], flow_vec[offset:offset+level_dim], n_patches, device)
        else:
            # Raw: embeddings already in full space; n_patches from n_patches_list
            assert n_patches_list is not None, f"n_patches_list required for raw (non-PCA) level {i}"
            n_patches = n_patches_list[i]
            n_comp = level_dim // n_patches
            emb = flow_vec[offset:offset+level_dim].reshape(n_patches, n_comp)
            emb = emb.to(device).unsqueeze(0)
        pos = make_grid_pos(n_patches, device)
        states.append(PatchState(emb=emb, pos=pos,
                                  non_empty=torch.ones(1, n_patches, device=device),
                                  mae_mask=torch.ones(n_patches, device=device)))
        offset += level_dim
    return states

def batch_patch_states(per_sample):
    """Batch a list of per-sample PatchState lists into one batched list."""
    return [
        PatchState(emb=torch.cat([s[li].emb for s in per_sample], dim=0),
                   pos=per_sample[0][li].pos,
                   non_empty=torch.cat([s[li].non_empty for s in per_sample], dim=0),
                   mae_mask=per_sample[0][li].mae_mask)
        for li in range(len(per_sample[0]))
    ]

def build_enc_out(levels):
    hps = HierarchicalPatchState(levels=levels)
    return EncoderOutput(patches=hps, full_pos=levels[-1].pos,
                         full_non_empty=levels[-1].non_empty,
                         mae_mask=levels[-1].mae_mask)

In [ ]:
#| export
def generate(cfg: DictConfig):
    """Full generation pipeline logged to W&B.
    Always produces two grids: flow-generated and real (sanity check) embeddings
    both put through inverse PCA → HMEP → decoder."""
    import wandb
    from torchvision.utils import make_grid, save_image
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    cjprint(f'Generating on {device}', color='cyan')
    gen = cfg.generate
    flow_cfg = cfg.flow

    # --- level_dims from dataset ---
    source_scales = list(flow_cfg.source_scales)
    raw_df = flow_cfg.get('source_df', None)
    source_df = list(raw_df) if hasattr(raw_df, '__iter__') else raw_df
    n_levels_flow = len(source_scales)
    levels = [f'L{i}' for i in range(n_levels_flow)]
    if source_df: source_df = source_df[:n_levels_flow]
    paths = sorted(glob.glob(os.path.expandvars(os.path.expanduser(flow_cfg.embedding_glob))))
    assert paths, f'No embedding files found: {flow_cfg.embedding_glob}'
    dataset = EmbeddingDataset(paths, levels=levels)
    level_dims = dataset.level_dims
    n_samples = gen.get('n_samples', 64)
    print(f'  level_dims: {level_dims}')

    # --- PCA models ---
    pca_dir = Path(os.path.expandvars(os.path.expanduser(gen.pca_dir)))
    pca_models = {}
    for i in range(n_levels_flow):
        with open(pca_dir / f'pca_L{i}_n20.pkl', 'rb') as f:
            pca_models[i] = pickle.load(f)

    # --- flow model ---
    t_dim = flow_cfg.get('t_dim', 64)
    model_type = gen.get('flow_model_type', flow_cfg.get('model_type', 'per_level'))
    self_condition = flow_cfg.get('self_condition', False)
    if model_type == 'cross_level':
        flow_model = CrossLevelFlowModel(level_dims=level_dims, h_dim=flow_cfg.h_dim,
                                          n_layers=flow_cfg.n_layers,
                                          n_attn_layers=flow_cfg.get('n_attn_layers', 2),
                                          n_heads=flow_cfg.get('n_heads', 8),
                                          self_condition=self_condition, t_dim=t_dim)
    else:
        flow_model = PerLevelFlowModel(level_dims=level_dims, h_dim=flow_cfg.h_dim,
                                        n_layers=flow_cfg.n_layers,
                                        self_condition=self_condition, t_dim=t_dim)
    flow_ckpt = os.path.expandvars(os.path.expanduser(gen.flow_ckpt))
    flow_model = load_checkpoint(flow_model, flow_ckpt).to(device).eval()

    # --- sample generated embeddings ---
    n_steps = gen.get('n_steps', 100)
    x_gen = sample_source((n_samples, sum(level_dims)), device=device,
                          source_df=source_df, source_scales=source_scales, level_dims=level_dims)
    dt = 1.0 / n_steps
    with torch.no_grad():
        for step in range(n_steps):
            t = torch.full((n_samples,), step * dt, device=device)
            x_gen = x_gen + flow_model(x_gen, t) * dt
    print(f'  Flow samples: {x_gen.shape}')

    # --- real embeddings (same n_samples, random subset) ---
    idx = torch.randperm(len(dataset))[:n_samples]
    x_real = dataset.embeddings[idx].to(device)

    # --- HMEP + decoder setup (shared) ---
    m = cfg.model
    n_stages = len(list(m.depths))
    enc_dims = [int(m.embed_dim * 2**(n_stages-1-i)) for i in range(n_stages)]
    raw_ns = gen.get('hmep_n_summaries', None)
    n_summaries = tuple(raw_ns) if raw_ns is not None else None
    hmep = SwinMaskedEmbeddingPredictor(dims=enc_dims, n_summaries=n_summaries)
    hmep = load_checkpoint(hmep, os.path.expandvars(os.path.expanduser(gen.hmep_ckpt))).to(device).eval()

    decoder = SwinDecoder(img_height=cfg.data.image_size, img_width=cfg.data.image_size,
                           patch_h=m.patch_h, patch_w=m.patch_w,
                           out_channels=cfg.data.in_channels,
                           embed_dim=m.embed_dim, depths=list(m.dec_depths),
                           num_heads=list(m.dec_num_heads), window_size=m.window_size,
                           mlp_ratio=m.mlp_ratio, drop_path_rate=0.0)
    decoder = load_checkpoint(decoder, os.path.expandvars(os.path.expanduser(gen.decoder_ckpt))).to(device).eval()

    fine_levels = list(range(n_levels_flow, n_stages))
    fine_dims   = [enc_dims[li] for li in fine_levels]
    fine_n_patches = [4**li for li in fine_levels]

    def run_pipeline(x):
        """inverse PCA → HMEP → decoder for a batch of PCA vectors x."""
        states = [build_patch_states(x[b], pca_models, level_dims, device) for b in range(len(x))]
        all_levels = batch_patch_states(states)
        for n_p, dim in zip(fine_n_patches, fine_dims):
            pos = make_grid_pos(n_p, device)
            all_levels.append(PatchState(emb=torch.zeros(len(x), n_p, dim, device=device),
                                          pos=pos,
                                          non_empty=torch.ones(len(x), n_p, device=device),
                                          mae_mask=torch.ones(n_p, device=device)))
        with torch.no_grad():
            hmep_preds, _ = hmep(build_enc_out(all_levels), mask_ratio=0)
        for j, (n_p, dim) in enumerate(zip(fine_n_patches, fine_dims)):
            li = n_levels_flow + j
            all_levels[li] = PatchState(emb=hmep_preds[li], pos=make_grid_pos(n_p, device),
                                         non_empty=torch.ones(len(x), n_p, device=device),
                                         mae_mask=torch.ones(n_p, device=device))
        with torch.no_grad():
            return decoder(build_enc_out(all_levels))

    recons_gen  = run_pipeline(x_gen)
    recons_real = run_pipeline(x_real)
    print(f'  Recon shape: {recons_gen.shape}')

    grid_gen  = make_grid(binarize(recons_gen[:64]),  nrow=8, normalize=True)
    grid_real = make_grid(binarize(recons_real[:64]), nrow=8, normalize=True)

    out_dir = Path(os.path.expandvars(os.path.expanduser(gen.get('output_dir', 'outputs/generate'))))
    out_dir.mkdir(parents=True, exist_ok=True)
    save_image(grid_gen,  out_dir / 'generated_piano_rolls.png')
    save_image(grid_real, out_dir / 'real_piano_rolls.png')

    use_wandb = hasattr(cfg, 'wandb') and hasattr(cfg.wandb, 'flow_project')
    if use_wandb:
        wandb.init(project=cfg.wandb.flow_project, config=dict(cfg.generate),
                   name=f"{cfg.tag}_gen")
        wandb.log({'generated': wandb.Image(grid_gen,  caption='flow generated'),
                   'real':      wandb.Image(grid_real, caption='real (inverse PCA sanity)')})
        wandb.finish()
        print('Logged to W&B')
    print('Done.')

In [ ]:
#| export
#| eval: false
import hydra

@hydra.main(version_base=None, config_path='../configs', config_name='config_swin')
def generate_main(cfg: DictConfig):
    generate(cfg)

if __name__ == '__main__':
    generate_main()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()